# Wind and SOR 2D debug

Scaffold for the **week-1 spike** (docs/ROADMAP.md): does the SOR Poisson solve converge on a building mask, and does the corrected flow go *around* the building?

A 2D x-z slice, one cell thick in `y`, so the array is still `[z, y, x]`.

Three questions to answer:
1. Does it converge, and in how many iterations?
2. Is `div(u)` below tolerance in every air cell afterwards?
3. Do the vectors go around the block, or through it?

In [1]:
import importlib.util
import sys
from pathlib import Path

import numpy as np

# src/02_wind.py is not a valid module name, and the notebook may be run from
# any directory, so load it by path.
ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

spec = importlib.util.spec_from_file_location("wind_module", ROOT / "src" / "02_wind.py")
wind = importlib.util.module_from_spec(spec)
sys.modules["wind_module"] = wind
spec.loader.exec_module(wind)

print("loaded", ROOT / "src" / "02_wind.py")

loaded /Users/jasonnguyen/AI-harness/ai-harness/projects/ie402-voxel-air-dispersion/src/02_wind.py


In [2]:
# ---- grid: x-z slice, one cell thick in y --------------------------------
DZ, DY, DX = 2.0, 5.0, 5.0
nz, ny, nx = 50, 1, 100
shape = (nz, ny, nx)

solid = np.zeros(shape, dtype=bool)
solid[0:10, :, 40:50] = True   # 20 m tall, 50 m wide

# ---- seed: power-law inflow, zero inside the building --------------------
U_REF, Z_REF, P = 3.0, 10.0, 0.25
z_centres = (np.arange(nz) + 0.5) * DZ

u0 = np.broadcast_to(
    (U_REF * (z_centres / Z_REF) ** P)[:, None, None], shape
).copy()
v0 = np.zeros(shape)
w0 = np.zeros(shape)
u0[solid] = 0.0

print(f"u at 2 m  = {u0[0, 0, 0]:.2f} m/s")
print(f"u at 50 m = {u0[24, 0, 0]:.2f} m/s")

u at 2 m  = 1.69 m/s
u at 50 m = 4.46 m/s


In [3]:
# ---- question 2, part one: how bad is the divergence BEFORE correcting? --
div_before = wind.divergence(u0, v0, w0, spacing=DX)
air = ~solid

print(f"max |div| before = {np.abs(div_before[air]).max():.4e}")
print("Non-zero is expected: the seed field is not mass-consistent.")

max |div| before = 3.5222e-01
Non-zero is expected: the seed field is not mass-consistent.


In [4]:
# ---- the spike itself ----------------------------------------------------
# sor_poisson is still a stub and raises NotImplementedError. Implementing it
# here, in 2D, IS the week-1 spike. Keep this cell as the harness for it.

try:
    lam = wind.sor_poisson(div_before, omega=1.78, tolerance=1e-4)
    print("solved, lambda shape", lam.shape)
except NotImplementedError as exc:
    print("STILL A STUB:", exc)
    print("\nWhat to implement, from docs/spec.md and docs/ARCHITECTURE.md section 4:")
    print("  d2L/dx2 + d2L/dy2 + (a1/a2)^2 d2L/dz2 = R,   R = div of the seed")
    print("  SOR update with omega = 1.78")
    print("  face coefficients e,f,g,h,m,n = 0 on any face that is a wall")
    print("  stop when sum|L_new - L_old| < 1e-4")
    print("  then u = u0 + (1/2a1^2) dL/dx,  and likewise for v and w")
    print("\nRaise, do not return, if the iteration cap is reached (spec EC-5).")

STILL A STUB: Implement boundary conditions for the selected voxel domain

What to implement, from docs/spec.md and docs/ARCHITECTURE.md section 4:
  d2L/dx2 + d2L/dy2 + (a1/a2)^2 d2L/dz2 = R,   R = div of the seed
  SOR update with omega = 1.78
  face coefficients e,f,g,h,m,n = 0 on any face that is a wall
  stop when sum|L_new - L_old| < 1e-4
  then u = u0 + (1/2a1^2) dL/dx,  and likewise for v and w

Raise, do not return, if the iteration cap is reached (spec EC-5).


In [5]:
# ---- questions 2 and 3: run these once sor_poisson works -----------------
# u, v, w = wind.apply_correction(u0, v0, w0, lam, DZ, DY, DX)
#
# div_after = wind.divergence(u, v, w, spacing=DX)
# assert np.abs(div_after[air]).max() < 1e-3, "still divergent"
# assert np.all(u[solid] == 0.0), "flow inside a building"
#
# import matplotlib.pyplot as plt
# step = 3
# X, Z = np.meshgrid(np.arange(nx) * DX, np.arange(nz) * DZ)
# fig, ax = plt.subplots(figsize=(11, 4))
# ax.quiver(X[::step, ::step], Z[::step, ::step],
#           u[::step, 0, ::step], w[::step, 0, ::step])
# ax.contourf(X, Z, solid[:, 0, :], levels=[0.5, 1.5], colors=["#c9d1d9"])
# ax.set_xlabel("x (m)"); ax.set_ylabel("z (m)")
# ax.set_title("corrected wind on an x-z slice")
# plt.show()
#
# The spike passes when the vectors bend over and around the grey block
# instead of passing through it.